# AI-Based Student Performance Prediction and Early Warning System Using Machine Learning

## Academic Minor Project: Model Exploration and Analysis Notebook

### Project Objective
This notebook documents the exploratory data analysis (EDA), feature engineering, Learning Behavior Index (LBI) calculation, model training, evaluation, and comparative benchmarking of four regression algorithms:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **Random Forest Regressor**
4. **Support Vector Regression (SVR)**

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.feature_engineering import calculate_lbi, add_engineered_features
from src.preprocessing import prepare_training_data, StudentDataPreprocessor
from src.recommendations import classify_risk, identify_weak_areas, generate_recommendations

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Environment initialized successfully.")

## 1. Load and Inspect Student Dataset

In [ ]:
csv_path = ROOT_DIR / "student_performance.csv"
df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
display(df.head())
df.info()

## 2. Statistical Summary & Missing Value Verification

In [ ]:
print("Missing values per feature:")
print(df.isnull().sum())

print("\nSummary Statistics:")
display(df.describe().T)

## 3. Correlation Heatmap Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="Blues", cbar=True)
plt.title("Correlation Heatmap: Academic & Behavioral Features vs Final Score", fontsize=13, fontweight="bold")
plt.show()

## 4. Learning Behavior Index (LBI) Distribution
$$\text{LBI} = 0.30 \times \text{Quiz} + 0.25 \times \text{Assignment} + 0.20 \times \text{Attendance} + 0.15 \times \text{Study Hours}_{\text{norm}} + 0.10 \times \text{LMS Activity}_{\text{norm}}$$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["lbi"], kde=True, color="purple", ax=ax1)
ax1.set_title("Distribution of Learning Behavior Index (LBI)", fontweight="bold")
ax1.set_xlabel("LBI (0–100)")

sns.scatterplot(x="lbi", y="final_score", data=df, hue="final_score", palette="viridis", ax=ax2)
ax2.set_title("LBI vs Final Score Relationship", fontweight="bold")
ax2.set_xlabel("Learning Behavior Index (LBI)")
ax2.set_ylabel("Final Examination Score (%)")
plt.tight_layout()
plt.show()

## 5. Model Training & Comparative Evaluation
Training 4 regression algorithms using 80/20 train/test split and standardized scaling.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Prepare data through preprocessing pipeline
X_train, X_test, y_train, y_test, preprocessor = prepare_training_data(
    df=df, target_column="final_score", test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    "Support Vector Regression (SVR)": SVR(kernel="rbf", C=15.0, epsilon=0.2),
}

benchmark = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    benchmark[name] = {
        "MAE": round(mean_absolute_error(y_test, preds), 4),
        "RMSE": round(root_mean_squared_error(y_test, preds), 4),
        "R2": round(r2_score(y_test, preds), 4),
    }

benchmark_df = pd.DataFrame(benchmark).T
print("Model Evaluation Benchmark Table:")
display(benchmark_df)

## 6. Visualizing Model Benchmark Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

benchmark_df["R2"].plot(kind="bar", ax=axes[0], color="#2563eb", edgecolor="black")
axes[0].set_title("R² Score (Higher is Better)", fontweight="bold")
axes[0].set_ylim(0.85, 1.0)
axes[0].set_ylabel("R²")
axes[0].tick_params(axis="x", rotation=30)

benchmark_df["MAE"].plot(kind="bar", ax=axes[1], color="#059669", edgecolor="black")
axes[1].set_title("MAE (Lower is Better)", fontweight="bold")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=30)

benchmark_df["RMSE"].plot(kind="bar", ax=axes[2], color="#d97706", edgecolor="black")
axes[2].set_title("RMSE (Lower is Better)", fontweight="bold")
axes[2].set_ylabel("RMSE")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 7. Sample Inference, Risk Classification & Recommendation Pipeline Test

In [ ]:
sample_student = {
    "previous_gpa": 6.2,
    "internal_marks": 52.0,
    "previous_subject_marks": 55.0,
    "quiz_marks": 50.0,
    "assignment_marks": 65.0,
    "attendance": 68.0,
    "study_hours": 1.8,
    "assignment_completion": 60.0,
    "quiz_participation": 70.0,
    "learning_activity": 45.0,
}

from src.prediction import StudentPredictor
predictor = StudentPredictor(ROOT_DIR / "models" / "best_model.pkl", ROOT_DIR / "models" / "preprocessor.pkl")
result = predictor.predict(sample_student)

print("=" * 45)
print("           PREDICTION RESULT")
print("=" * 45)
print(f"Predicted Final Score   : {result['predicted_score']}%")
print(f"Learning Behavior Index : {result['lbi']}")
print(f"Risk Level              : {result['risk_level']}")
print("\nDetected Weak Areas:")
for wa in result["weak_areas"]:
    print(f" - {wa['title']}: {wa['current_value']} (Threshold: {wa['threshold']})")
print("\nPersonalized Recommendations:")
for rec in result["recommendations"]:
    print(f"  [v] {rec}")
print("=" * 45)